# Building Generative Pretrained Transformer (GPT) From Scratch

In [1]:
# Importing Dependencies
from transformers import GPT2Model
import tiktoken
import warnings
import torch
import yaml

In [2]:
# Filtering out potential warnings
warnings.filterwarnings(action="ignore")

In [3]:
# Loading the configuration
with open(file="../Configuration/config.yml") as yaml_file:
    config=yaml.safe_load(stream=yaml_file)
    
# Extracting the configuration for GPT (Original)
llm_config=config["gpt"]["original"]

# Displaying the configurations for GPT (Original)
llm_config

{'vocab_size': 50257,
 'context_length': 1024,
 'emb_dim': 1600,
 'n_heads': 25,
 'n_layers': 48,
 'drop_rate': 0.1,
 'qkv_bias': True}

In [4]:
# Defining a device name
device_name="cuda" if torch.cuda.is_available() else "cpu"

# Specifying the device
device=torch.device(device=device_name)

# Making sure the GPU is being utilized
assert torch.cuda.is_available()

### Layer Normalization

In [5]:
# Designing a custom Layer Normalization (LayerNorm)
class LayerNorm(torch.nn.Module):
    # Defining instance attribute(s)
    def __init__(self, emb_dim: int, eps: float=1e-5):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the shift parameter
        self.shift=torch.nn.Parameter(data=torch.zeros(emb_dim))
        
        # Defining the scale parameter
        self.scale=torch.nn.Parameter(data=torch.ones(emb_dim))
        
        # Defining the epsilon parameter
        self.eps=eps
    
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the average of the tensor
        mean=x.mean(dim=-1, keepdim=True)
        
        # Calculating the variance of the tensor
        var=x.var(dim=-1, unbiased=False, keepdim=True)
        
        # Normalizing the values
        norm_x=(x-mean)/torch.sqrt(input=var+self.eps)
        
        # Adding the scale and shift parameters
        norm_x=(norm_x*self.scale)+self.shift
        
        # Returning the normalized values
        return norm_x

In [6]:
# Creating a sample data
sample_data=torch.randn(size=[2, 16, llm_config["emb_dim"]])

# Applying normalization
LayerNorm(emb_dim=llm_config["emb_dim"])(sample_data)

tensor([[[-2.7210e-03, -2.6556e-01, -4.8893e-01,  ...,  2.5463e-01,
           1.8490e+00,  1.1207e-01],
         [-5.3844e-01, -1.1551e+00, -1.2924e+00,  ...,  1.4890e-01,
          -9.4732e-02,  6.0196e-01],
         [-9.8710e-01, -5.2699e-01,  1.2241e+00,  ...,  4.1689e-01,
           1.3594e+00,  8.9085e-03],
         ...,
         [ 1.1127e+00, -1.5846e+00,  1.2441e-01,  ...,  1.4709e+00,
          -1.7199e+00,  2.2539e-01],
         [ 8.8324e-01,  6.6668e-01, -2.8581e-01,  ..., -7.5195e-01,
          -3.6650e-01, -6.8100e-01],
         [-1.2189e+00, -1.4733e+00, -3.2396e-01,  ..., -5.6199e-01,
           4.7065e-01, -8.0139e-01]],

        [[ 1.3184e+00,  7.9193e-01, -5.3353e-01,  ..., -1.2246e+00,
          -1.2288e-03,  1.4790e+00],
         [-1.1865e-01, -4.4066e-01, -9.0618e-01,  ...,  5.5864e-01,
           9.4191e-01,  2.1800e+00],
         [ 1.6627e+00, -2.5944e+00, -7.2402e-01,  ..., -1.0726e+00,
           1.5148e+00,  4.4176e-01],
         ...,
         [-1.6848e+00, -1

In [7]:
# Checking if both tensors are same
torch.allclose(LayerNorm(emb_dim=llm_config["emb_dim"])(sample_data),
               torch.nn.LayerNorm(normalized_shape=llm_config["emb_dim"])(sample_data))

True

### Gaussian Error Linear Unit (GELU) Activation Function

In [8]:
# Designing a custom GELU activation function
class GELU(torch.nn.Module):
    # Defining instance attributes
    def __init__(self):
        # Inheriting methods and attributes
        super().__init__()
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the gelu
        gelu=0.5*x*(1+torch.tanh(input=torch.sqrt(torch.tensor(data=2/torch.pi)) * (x+(0.044715*torch.pow(input=x, exponent=3)))))
        
        # Returning the gelu
        return gelu

In [9]:
# Applying the activation function
GELU()(sample_data)

tensor([[[ 4.3626e-05, -1.0608e-01, -1.5407e-01,  ...,  1.5942e-01,
           1.8437e+00,  6.4400e-02],
         [-1.6199e-01, -1.4008e-01, -1.2330e-01,  ...,  6.4445e-02,
          -5.6496e-02,  4.0828e-01],
         [-1.6020e-01, -1.5848e-01,  1.0307e+00,  ...,  2.5065e-01,
           1.1786e+00, -5.9720e-03],
         ...,
         [ 9.0905e-01, -8.8829e-02,  4.8021e-02,  ...,  1.3033e+00,
          -7.2979e-02,  1.0867e-01],
         [ 7.5655e-01,  5.3182e-01, -1.0496e-01,  ..., -1.7000e-01,
          -1.2651e-01, -1.6837e-01],
         [-1.4046e-01, -1.0902e-01, -1.1356e-01,  ..., -1.5817e-01,
           3.3717e-01, -1.6999e-01]],

        [[ 1.1890e+00,  6.3385e-01, -1.5163e-01,  ..., -1.4399e-01,
           1.7930e-02,  1.3646e+00],
         [-4.6051e-02, -1.4442e-01, -1.6459e-01,  ...,  4.4035e-01,
           8.4607e-01,  2.2693e+00],
         [ 1.5760e+00, -1.0721e-02, -1.7003e-01,  ..., -1.4959e-01,
           1.4087e+00,  2.8417e-01],
         ...,
         [-7.7844e-02, -8

In [10]:
# Checking if both tensors are same
torch.allclose(input=GELU()(sample_data),
               other=torch.nn.GELU(approximate="tanh")(sample_data))

True

### Multi Head Attention (MHA) Mechanism

#### Inputs:

- n_heads: The number of attention heads
- qkv_bias: The flag whether to apply bias for vector calculation
- d_model: The dimension (input & output) used for vector calculation
- context_length: The maximum number of tokens an LLM can process
- dropout_proba: The probability threshold used for regularization

#### Outputs:

- Context vectors with regularized causal masked attention weights

In [11]:
# Designing a multi head attention (MHA) mechanism
class MultiHeadAttention(torch.nn.Module):
    # Defining the instance attribute(s)
    def __init__(self,
                 d_model: int,
                 n_heads: int,
                 qkv_bias: bool,
                 context_length: int,
                 dropout_proba: float):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Asserting the dimension to be divisible by the number of attention heads
        assert d_model % n_heads == 0
        
        # Defining the causal masked attention mechanism
        self.register_buffer(name="mask", tensor=torch.triu(input=torch.ones(size=[context_length, context_length], dtype=torch.bool), diagonal=1))
        
        # Defining the linear layer for the calculation of query, key, and value (qkv) vectors
        self.W_qkv=torch.nn.Linear(in_features=d_model, out_features=d_model*3, bias=qkv_bias)
        
        # Defining the linear projections layer that are necessary for LLMs
        self.linear_proj=torch.nn.Linear(in_features=d_model, out_features=d_model)
        
        # Defining the regularization
        self.dropout=torch.nn.Dropout(p=dropout_proba)
        
        # Defining the number of dimensions for the attention heads
        self.head_dim=d_model // n_heads
        
        # Defining the dimension (input & output)
        self.d_model=d_model
        
        # Defining the number of attention heads
        self.n_heads=n_heads
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Extracting the number of batches, and tokens
        n_batches, n_tokens, _ = x.shape
        
        # Instantiating the query, key, and value vectors
        qkv=self.W_qkv(x)
        
        # Unpacking the tensor to extract the query, key and value vectors
        queries, keys, values=torch.chunk(input=qkv, chunks=3, dim=-1)
        
        # Reshaping the query vectors from [b, t, d] to [b, t, h, d]
        queries=queries.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Reshaping the value vectors from [b, t, d] to [b, t, h, d]
        values=values.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Reshaping the key vectors from [b, t, d] to [b, t, h, d]
        keys=keys.view(n_batches, n_tokens, self.n_heads, self.head_dim)
        
        # Tranposing the query vectors from [b, t, h, d] to [b, h, t, d]
        queries=queries.transpose(1, 2)
        
        # Tranposing the value vectors from [b, t, h, d] to [b, h, t, d]
        values=values.transpose(1, 2)
        
        # Tranposing the key vectors from [b, t, h, d] to [b, h, t, d]
        keys=keys.transpose(1, 2)
        
        # Calculating the attentions scores
        attn_scores=queries @ keys.transpose(-2, -1)
        
        # Instantiating the masking for the causal attention mechanism
        masking=self.mask[:n_tokens, :n_tokens]
        
        # Applying the masking to attention scores
        attn_scores_masked=attn_scores.masked_fill(mask=masking, value=-torch.inf)
        
        # Calculating the regularized attention weights
        attn_weights=self.dropout(torch.softmax(input=attn_scores_masked/self.head_dim**0.5, dim=-1))
        
        # Calculating the context vectors
        context_vectors=(attn_weights @ values).transpose(1, 2)
        
        # Reshaping the context vectors and applying linear projections
        context_vectors=self.linear_proj(context_vectors.contiguous().view(n_batches, n_tokens, self.d_model))
        
        # Returning the context vectors
        return context_vectors

In [12]:
# Creating a dictionary of keyword arguments
llm_kwargs=dict(d_model=llm_config["emb_dim"],
                n_heads=llm_config["n_heads"],
                qkv_bias=llm_config["qkv_bias"],
                context_length=llm_config["context_length"],
                dropout_proba=llm_config["drop_rate"])

# Instantiating the MHA
mha=MultiHeadAttention(**llm_kwargs)

# Sending the sample data to MHA
output=mha(sample_data)

# Displaying the shape and output
display(output.shape, output)

torch.Size([2, 16, 1600])

tensor([[[ 1.1210e-01, -2.0493e-01, -1.6703e-02,  ...,  1.0721e-01,
           2.4595e-02, -9.9272e-02],
         [-1.9894e-01, -2.7683e-01, -3.7893e-02,  ...,  4.5425e-03,
           1.9835e-01,  1.2631e-01],
         [-1.6175e-02, -1.9313e-01,  4.5357e-02,  ..., -1.3409e-01,
           1.2416e-01, -2.4716e-01],
         ...,
         [ 2.2336e-03, -1.0897e-01,  2.4276e-02,  ..., -6.1244e-02,
           4.0711e-02, -1.8670e-01],
         [-1.4830e-01, -1.0314e-01, -4.0632e-02,  ..., -7.7758e-02,
           8.3530e-02, -1.7430e-01],
         [-5.4218e-02, -1.2045e-01, -5.3746e-02,  ..., -6.2771e-02,
           3.5443e-02, -1.3194e-01]],

        [[ 7.1773e-01, -2.2272e-01, -1.6759e-01,  ..., -2.3287e-01,
           1.8030e-01, -4.2010e-01],
         [ 1.8386e-01,  9.1431e-03,  1.4252e-01,  ..., -3.0645e-01,
           4.4599e-01,  7.4118e-02],
         [ 5.5669e-02, -3.4838e-02, -4.3147e-03,  ..., -1.9074e-01,
           4.7862e-01,  3.1070e-02],
         ...,
         [ 2.4056e-02,  2

### Feed Forward Neural Network

In [13]:
# Designing the Feed Forward Neural Network
class FeedForward(torch.nn.Module):
    # Defining the instance attribute(s)
    def __init__(self, emb_dim: int):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the input layer
        input_layer=torch.nn.Linear(in_features=emb_dim, out_features=emb_dim*4)
        
        # Defining the output layer
        output_layer=torch.nn.Linear(in_features=emb_dim*4, out_features=emb_dim)
        
        # Defining the activation function
        gelu=GELU()
        
        # Defining a sequential neural network
        self.layers=torch.nn.Sequential(input_layer, gelu, output_layer)
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculating the hidden states
        hidden_states=self.layers(x)
        
        # Returning the hidden states
        return hidden_states

### Transformer (Decoder Only)

In [14]:
# Designing a transformer architecture
class TransformerBlock(torch.nn.Module):
    # Defining the instance attributes
    def __init__(self, config: dict):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Creating a dictionary of keyword arguments
        mha_kwargs=dict(d_model=config["emb_dim"],
                        n_heads=config["n_heads"],
                        qkv_bias=config["qkv_bias"],
                        context_length=config["context_length"],
                        dropout_proba=config["drop_rate"])
        
        # Instantiating the multi head attention
        self.mha=MultiHeadAttention(**mha_kwargs)
        
        # Instantiating the feed forward neural network
        self.ff=FeedForward(emb_dim=config["emb_dim"])
        
        # Instantiating the layer normalization
        self.layer_norm_mha=LayerNorm(emb_dim=config["emb_dim"])
        
        # Instantiating the layer normalization
        self.layer_norm_ff=LayerNorm(emb_dim=config["emb_dim"])
        
        # Instantiating the regularization
        self.dropout=torch.nn.Dropout(p=config["drop_rate"])
        
    # Defining the forward instance method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initializing the shortcut tensor
        shortcut=x
        
        # Applying normalization
        x=self.layer_norm_mha(x)
        
        # Calculating the context vectors
        x=self.mha(x)
        
        # Applying regularization
        x=self.dropout(x)
        
        # Updating the tensors with shortcut values
        x=x+shortcut
        
        # Reinitializing the shortcut with updated values
        shortcut=x
        
        # Applying normalization
        x=self.layer_norm_ff(x)
        
        # Calculating the hidden states
        x=self.ff(x)
        
        # Applying regularization
        x=self.dropout(x)
        
        # Updating the tensors with shortcut values
        x=x+shortcut
        
        # Returning the values
        return x

### GPT Model

In [15]:
# Designing the GPT model
class GPTModel(torch.nn.Module):
    # Defining instance attributes
    def __init__(self, config: dict):
        # Inheriting the methods and attributes
        super().__init__()
        
        # Defining the token embedding layer
        self.tok_emb_layer=torch.nn.Embedding(num_embeddings=config["vocab_size"], embedding_dim=config["emb_dim"])
        
        # Defining the positional (absolute) embedding layer
        self.pos_emb_layer=torch.nn.Embedding(num_embeddings=config["context_length"], embedding_dim=config["emb_dim"])
        
        # Defining the transformer blocks
        self.trf_blocks=torch.nn.Sequential(*[TransformerBlock(config=config) for _ in range(config["n_layers"])])
        
        # Defining the linear projections that are necessary for LLMs
        self.lm_head=torch.nn.Linear(in_features=config["emb_dim"], out_features=config["vocab_size"], bias=False)
        
        # Defining the layer normalization
        self.final_layer_norm=LayerNorm(emb_dim=config["emb_dim"])
        
        # Defining the regularization
        self.dropout=torch.nn.Dropout(p=config["drop_rate"])
        
    # Defining the forward method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Unpacking the values of the tensor
        _, seq_len=x.shape
        
        # Calculating the token embeddings based on the inputs
        tok_embs=self.tok_emb_layer(x)
        
        # Calculating the positional (absolute) embeddings based on the sequence length
        pos_embs=self.pos_emb_layer(torch.arange(seq_len, device=x.device))
        
        # Calculating the combined embeddings
        x=tok_embs+pos_embs
        
        # Applying regularization
        x=self.dropout(x)
        
        # Calculating hidden states
        x=self.trf_blocks(x)
        
        # Applying the final normalization
        x=self.final_layer_norm(x)
        
        # Mapping hidden states to vocabulary logits
        x=self.lm_head(x)
        
        # Returning the logits
        return x

In [16]:
# Loading the tokenizer
tokenizer=tiktoken.get_encoding(encoding_name="gpt2")

# Creating a sample sentence
sample="I like watching movies"

# Creating a batch
sample_tensor=torch.tensor(data=tokenizer.encode(text=sample, allowed_special={"<|endoftext|>"})).unsqueeze(dim=0)

# Instantiating the GPT model
model=GPTModel(config=llm_config)

# Calculating the logits
logits=model(sample_tensor)

# Displaying the logits
logits

tensor([[[-1.0829,  0.3690,  0.1636,  ..., -0.1112, -0.0324,  0.3554],
         [-0.7253,  0.5133,  0.7214,  ..., -0.5136,  0.5212,  0.0183],
         [-0.4695, -0.0469,  0.0187,  ..., -0.3498,  0.1487, -0.0051],
         [-0.0617,  0.7611, -0.2727,  ..., -0.4536, -0.0375,  0.6913]]],
       grad_fn=<UnsafeViewBackward0>)

#### Generating text

In [17]:
# Defining a function to generate text
def generate_text(model,
                  token_ids,
                  max_new_tokens,
                  context_length,
                  temperature=0.0,
                  top_k=None,
                  eos_id=None):
    # Looping through the range of maximum new tokens
    for _ in range(max_new_tokens):
        # Cropping the sequence length to match the context length
        cropped_token_ids=token_ids[:, -context_length:]

        # Disabling gradients
        with torch.no_grad():
            # Making predictions
            logits=model(cropped_token_ids)

        # Extracting the embeddings for the last step
        logits=logits[:, -1, :]

        # Creating a condition based on top K
        if top_k is not None:
            # Extracting the top K logits
            top_logits, _ = torch.topk(input=logits, k=top_k)

            # Extracting the minimum value from the logits
            min_logit=top_logits[:, -1:]

            # Updating the logits
            logits=torch.where(condition=logits<min_logit, input=torch.tensor(data=float("-inf")).to(device=logits.device), other=logits)

        # Creating a condition based on the temperature scaling
        if temperature>0.0:
            # Dividing the logits by temperature
            logits=logits/temperature

            # Calculating the probabilities
            probas=torch.softmax(input=logits, dim=-1)

            # Extracting the next token ID
            next_token_id=torch.multinomial(input=probas, num_samples=1)
        else:
            # Extracting the next token ID with the highest probability
            next_token_id=torch.argmax(input=logits, dim=-1, keepdim=True)

        # Creating a condition based on the end of sequence token ID
        if next_token_id==eos_id:
            # Stopping text generation
            break
        
        # Concatenating the next token ID to the current token IDs
        token_ids=torch.cat(tensors=(token_ids, next_token_id), dim=1)

    # Returning the token IDs
    return token_ids

In [ ]:
# Fixing the seed for reproducibility
torch.manual_seed(123)

# Defining the end of sequence ID (EOS)
eos_id=tokenizer.encode(text="<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

# Defining a sample text
sample_text="He never wanted to"

# Converting the text into token IDs
sample_token_ids=torch.tensor(tokenizer.encode(text=sample_text, allowed_special={"<|endoftext|>"})).unsqueeze(dim=0)

# Generating the next tokens
generated_token_ids=generate_text(model=model,
                                  token_ids=sample_token_ids,
                                  max_new_tokens=5,
                                  context_length=llm_config["context_length"],
                                  temperature=0.5,
                                  top_k=50,
                                  eos_id=eos_id)

# Displaying the generated token IDs
generated_token_ids

tensor([[ 1544,  1239,  2227,   284, 17383, 46350, 47517, 42397,  2451]])

In [19]:
# Printing out the generated token IDs
print(tokenizer.decode(tokens=generated_token_ids.squeeze().tolist()))

He never wanted to retained Drill refinement contended Sw


### Loading Pretrained GPT-2 Weights

In [20]:
# Defining a function to download and load the weights
def download_and_load_gpt2(model_size, models_dir=None):
    # Mapping the model sizes to Hugging Face model IDs (models_dir is kept for book compatibility; unused)
    allowed_sizes={"124M": "openai-community/gpt2",
                   "355M": "openai-community/gpt2-medium",
                   "774M": "openai-community/gpt2-large",
                   "1558M": "openai-community/gpt2-xl"}

    # Asserting the model size to be supported
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {tuple(allowed_sizes)}")

    # Downloading the PyTorch model and converting its weights to NumPy
    hf_model=GPT2Model.from_pretrained(allowed_sizes[model_size])
    sd={k: v.cpu().numpy() for k, v in hf_model.state_dict().items()}

    # Extracting the hyperparameters from the Hugging Face configuration
    cfg=hf_model.config
    settings={"n_vocab": cfg.vocab_size, "n_ctx": cfg.n_positions,
              "n_embd": cfg.n_embd, "n_head": cfg.n_head, "n_layer": cfg.n_layer}

    # Mapping the root weights
    params={"wte": sd["wte.weight"], "wpe": sd["wpe.weight"],
            "g": sd["ln_f.weight"], "b": sd["ln_f.bias"], "blocks": []}

    # Mapping the layer blocks
    for i in range(settings["n_layer"]):
        p=f"h.{i}."
        params["blocks"].append({
            "attn": {"c_attn": {"w": sd[p+"attn.c_attn.weight"], "b": sd[p+"attn.c_attn.bias"]},
                     "c_proj": {"w": sd[p+"attn.c_proj.weight"], "b": sd[p+"attn.c_proj.bias"]}},
            "ln_1": {"g": sd[p+"ln_1.weight"], "b": sd[p+"ln_1.bias"]},
            "ln_2": {"g": sd[p+"ln_2.weight"], "b": sd[p+"ln_2.bias"]},
            "mlp": {"c_fc": {"w": sd[p+"mlp.c_fc.weight"], "b": sd[p+"mlp.c_fc.bias"]},
                    "c_proj": {"w": sd[p+"mlp.c_proj.weight"], "b": sd[p+"mlp.c_proj.bias"]}},
        })

    # Returning the settings and parameters
    return settings, params


In [ ]:
# Defining a function to copy a NumPy array into a parameter after checking the shapes
def assign(left, right):
    # Asserting the shapes to match
    if left.shape!=right.shape:
        raise ValueError(f"Shape mismatch. Left: {tuple(left.shape)}, Right: {right.shape}")

    # Returning the new parameter
    return torch.nn.Parameter(torch.tensor(right, dtype=left.dtype))

# Defining a function to load the GPT-2 weights into the GPT model
def load_weights_into_gpt(gpt, params):
    # Loading the embedding weights
    gpt.pos_emb_layer.weight=assign(gpt.pos_emb_layer.weight, params["wpe"])
    gpt.tok_emb_layer.weight=assign(gpt.tok_emb_layer.weight, params["wte"])

    # Looping through the transformer blocks
    for block, p in zip(gpt.trf_blocks, params["blocks"]):
        # Loading the fused query, key, and value weights (Conv1D stores [in, out], Linear expects [out, in])
        block.mha.W_qkv.weight=assign(block.mha.W_qkv.weight, p["attn"]["c_attn"]["w"].T)
        block.mha.W_qkv.bias=assign(block.mha.W_qkv.bias, p["attn"]["c_attn"]["b"])

        # Loading the attention output projection weights
        block.mha.linear_proj.weight=assign(block.mha.linear_proj.weight, p["attn"]["c_proj"]["w"].T)
        block.mha.linear_proj.bias=assign(block.mha.linear_proj.bias, p["attn"]["c_proj"]["b"])

        # Loading the feed forward neural network weights
        block.ff.layers[0].weight=assign(block.ff.layers[0].weight, p["mlp"]["c_fc"]["w"].T)
        block.ff.layers[0].bias=assign(block.ff.layers[0].bias, p["mlp"]["c_fc"]["b"])
        block.ff.layers[2].weight=assign(block.ff.layers[2].weight, p["mlp"]["c_proj"]["w"].T)
        block.ff.layers[2].bias=assign(block.ff.layers[2].bias, p["mlp"]["c_proj"]["b"])

        # Loading the layer normalization weights
        block.layer_norm_mha.scale=assign(block.layer_norm_mha.scale, p["ln_1"]["g"])
        block.layer_norm_mha.shift=assign(block.layer_norm_mha.shift, p["ln_1"]["b"])
        block.layer_norm_ff.scale=assign(block.layer_norm_ff.scale, p["ln_2"]["g"])
        block.layer_norm_ff.shift=assign(block.layer_norm_ff.shift, p["ln_2"]["b"])

    # Loading the final layer normalization weights
    gpt.final_layer_norm.scale=assign(gpt.final_layer_norm.scale, params["g"])
    gpt.final_layer_norm.shift=assign(gpt.final_layer_norm.shift, params["b"])

    # Loading the output head weights (GPT-2 ties them to the token embeddings)
    gpt.lm_head.weight=assign(gpt.lm_head.weight, params["wte"])

In [22]:
# Downloading the GPT-2 (Small) weights
settings, params=download_and_load_gpt2(model_size="124M")

# Extracting the matching configuration for GPT-2 (Small)
gpt2_config=config["gpt"]["small"]

# Instantiating the GPT model
gpt2=GPTModel(config=gpt2_config)

# Loading the pretrained weights
load_weights_into_gpt(gpt=gpt2, params=params)

# Moving the model to the device and switching to evaluation mode
gpt2=gpt2.to(device=device).eval()

# Displaying the settings
settings

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}

In [23]:
# Fixing the seed for reproducibility
torch.manual_seed(123)

# Converting the text into token IDs
pretrained_token_ids=torch.tensor(tokenizer.encode(text="Every effort moves you")).unsqueeze(dim=0).to(device=device)

# Generating the next tokens with the pretrained model
pretrained_token_ids=generate_text(model=gpt2,
                                   token_ids=pretrained_token_ids,
                                   max_new_tokens=25,
                                   context_length=gpt2_config["context_length"],
                                   temperature=1.5,
                                   top_k=50,
                                   eos_id=eos_id)

# Printing out the generated text
print(tokenizer.decode(tokens=pretrained_token_ids.squeeze().tolist()))

Every effort moves you as far as the hand can go until the end of your turn unless something interrupts your control flow. As you may observe I
